# 2.3 Image enhancement

Image enhancement is the process of adjusting a digital image so the resultant one is more suitable for further image analysis (edge detection, feature extraction, segmentation, etc.), in other words, **its goal is to improve the contrast and brightness of the image**.

There are three typical operations for enhancing images. We have already explored one of them in notebook *2.1 IP tools*: (linear) Look-Up Tables (LUTs). In this notebook we will play with a variant of LUTs and other two operations:

- Non-linear look-up tables (<a href="#231">Section 2.3.1</a>).
- Histogram equalization (<a href="#232">Section 2.3.2</a>).
- Contrast Limited Adaptive Histogram Equalization (CLAHE) (<a href="#233">Section 2.3.3</a>).

Also, some color-space conversions are going to be needed. If you are not familiar with the YCrCb color space, **Appendix 2: Color spaces** contains the information you need to know about it.


## Problem context - Implementing enhancement techniques for an image editor tool

We have all tried an image editor tool, sometimes without even knowing it! For example, modern smartphones already include an application for applying filters to images, cut them, modify their contrast, brightness, color temperature, etc.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_processing/images/GIMP-logo.png" width="200"></center>

One example of open source tool is the GNU Image Manipulation Program (GIMP). Quoting some words from its [website](https://www.gimp.org/):  

> GIMP is a cross-platform image editor available for GNU/Linux, OS X, Windows and more operating systems. It is free software, you can change its source code and distribute your changes.
> Whether you are a graphic designer, photographer, illustrator, or scientist, GIMP provides you with sophisticated tools to get your job done. You can further enhance your productivity with GIMP thanks to many customization options and 3rd party plugins.


In this case we were contacted by UMA for implementing two techniques to be included in their own image editor tool! Concretely, we were asked to develop and test two methods that are also part of GIMP: [**gamma correction**](https://docs.gimp.org/2.10/en/gimp-tool-levels.html) and [**equalize**](https://docs.gimp.org/2.10/en/gimp-layer-equalize.html).

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 02. Image processing'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
from ipywidgets import interactive, fixed, widgets
matplotlib.rcParams['figure.figsize'] = (20.0, 20.0)

images_path = './images/'

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 02. Image processing


## 2.3.1 Non-linear look-up tables <a id=231></a>

**Gamma correction**, or often simply **gamma**, is a nonlinear operation used to adjust the luminance or brightness levels of an image. In other words, it is the result of applying an (already defined) **non-linear LUT** in order to stretch or shrink image intensities.

In this way, the gamma LUT definition for grayscale images, where each pixel $i$ takes values in the range $[0 \dots 255]$, is:

$$LUT(i) = (\frac{i}{255})^{\gamma} * 255, \ \gamma \gt 0 $$

The following images illustrate the application of gamma correction for different values of $\gamma$.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_processing/images/gamma_theory.jpg" width="800" ></center>

The role of $\gamma$:

- $\gamma < 1$ : The image is lightened. Dark areas become brighter, enhancing shadow details.
- $\gamma = 1$ : No change is applied; the output is identical to the input.
- $\gamma > 1$ : The image is darkened. Bright areas become darker, which can reduce glare or overexposure.

### **<span style="color:green"><b><i>ASSIGNMENT 1: Applying non-linear LUTs</i></b></span>**

Your task is to develop the `lut_chart()` function, which takes as arguments the image to be enhanced and a gamma value for building the non-linear LUT. It will also display a chart containing the original image, the gamma-corrected one, the used LUT and the histogram of the resulting image.

As users from UMA will use color images, you will have to **implement it for color images**. This can be done by:
1. **transforming** an image in the BGR color space **to the YCrCb one**,
2. then, **applying gamma LUT only to first band** of the YCrCb space (that's because it contains pixel intensities and you can handle it like a gray image), and
3. finally, as matplotlib displays RGB images (if verbose is True), it should be **converted back**. Also, return the resultant image.

*Interesting functions:*
- [`np.copy()`](https://numpy.org/doc/stable/reference/generated/numpy.copy.html): method that returns a copy of the array provided as input.
- [`cv2.LUT()`](https://docs.opencv.org/master/d2/de8/group__core__array.html#gab55b8d062b7f5587720ede032d34156f): *function that  performs a look-up table transform of an array of arbitrary dimensions.*
- [`plt.hist()`](https://matplotlib.org/3.3.1/api/_as_gen/matplotlib.pyplot.hist.html) *function that computes and draws the histogram of an array.  [`numpy.ravel()`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.ravel.html) is a good helper here, since it converts a n-dimensional array into a flattened 1D array.*

In [ ]:
# ASSIGNMENT 1
# Implement a function that:
# -- converts the input image from the BGR to the YCrCb color space
# -- creates the gamma LUT
# -- applies the LUT to the original image
# -- displays in a 2x2 plot: the input image, the gamma-corrected one, the applied LUT and the resultant histogram if verbose = True
def lut_chart(image, gamma, verbose=False):
    """ Applies gamma correction to an image and shows the result.

        Args:
            image: Input image
            gamma: Gamma parameter
            verbose: Only show images if this is True

        Returns:
            out_image: Gamma image
    """

    #Transform image to YCrCb color space
    image = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    out_image = np.copy(image)

    # Define gamma correction LUT
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    # Apply LUT to first band of the YCrCb image
    out_image[:,:,0] = cv2.LUT(image[:,:,0], lut)

    if verbose:
        # Plot used LUT
        plt.subplot(2,2,3)
        plt.title('LUT')
        plt.plot(np.arange(256), lut)

        # Plot histogram of gray image after applying the LUT
        plt.subplot(2,2,4)
        plt.hist(out_image[:,:,0].ravel(), 256, [0, 256])
        plt.title('Histogram')

        # Reconvert image to RGB
        image = cv2.cvtColor(image, cv2.COLOR_YCrCb2RGB)
        out_image = cv2.cvtColor(out_image, cv2.COLOR_YCrCb2RGB)

        # Show the initial image
        plt.subplot(2,2,1)
        plt.imshow(image)
        plt.title('Original image')

        # Show the resultant one
        plt.subplot(2,2,2)
        plt.imshow(out_image)
        plt.title('LUT applied')

        plt.show()

    return out_image

You can use the next code to **test if results are correct**:

In [ ]:
image = np.array([[[10,60,20],[60,22,74],[72,132,2]],[[11,63,42],[36,122,27],[37,113,30]],[[1,6,2],[6,22,7],[7,13,0]]], dtype=np.uint8)
gamma = 2
print(lut_chart(image, gamma))

[[[  6 112 110]
  [  6 151 138]
  [ 29  68 120]]

 [[ 10 122 105]
  [ 27  87 101]
  [ 25  92 104]]

 [[  0 127 126]
  [  1 122 122]
  [  0 122 127]]]


<font color='blue'>**Expected output:**  </font>

    [[[  6 112 110]
      [  6 151 138]
      [ 29  68 120]]

     [[ 10 122 105]
      [ 27  87 101]
      [ 25  92 104]]

     [[  0 127 126]
      [  1 122 122]
      [  0 122 127]]]

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

In the interactive code cell below, **you are asked to** explore how your new `lut_chart()` function works with `gamma_1.jpg` (an underexposed image) and `gamma_2.jpeg` (an overexposed image). Then, **answer the following question** (you can take a look at the LUT and the resulting histogram):

- What is happening when the *gamma value* is modified?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>  

In [ ]:
# Create widget object
gamma_widget = widgets.FloatSlider(value=1, min=0.1, max=5, step=0.1, description='Gamma:')

#Read image
image = cv2.imread(images_path + 'gamma_2.jpeg',-1)

#Interact with your code!
interactive(lut_chart, image=fixed(image), gamma=gamma_widget, verbose=fixed(True))

interactive(children=(FloatSlider(value=1.0, description='Gamma:', max=5.0, min=0.1), Output()), _dom_classes=…

## 2.3.2 Histogram equalization <a id=232></a>

**Histogram equalization** is an image processing technique used to improve contrast in images. It operates by effectively spreading out the most frequent intensity values, i.e. stretching out the intensity range of the image so each possible pixel intensity appears the same number of times as every other value. This method usually increases the global contrast of images when its usable data is represented by close contrast values. This allows for areas of lower local contrast to gain a higher contrast.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_processing/images/equalization.png" width="300" /></center>

To put an example, the [**equalize**](https://docs.gimp.org/2.10/en/gimp-layer-equalize.html) command from GIMP applies histogram equalization. But... how is this equalization achieved?

- First it is calculated the PMF ([probability mass function](https://en.wikipedia.org/wiki/Probability_mass_function)) of all the pixels in the image. Basically, this is a normalization of the histogram.

- Next step involves calculation of CDF ([cumulative distributive function](https://en.wikipedia.org/wiki/Cumulative_distribution_function)), producing the LUT for histogram equalization.

- Finally, the obtained LUT is applied.

The figure below shows an example of applying histogram equalization to an image.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_processing/images/equalize_theory.png" width="600" /></center>

### **<span style="color:green"><b><i>ASSIGNMENT 2: Equalizing the histogram!</i></b></span>**

Similarly to the previous exercise, **you are asked to** develop a function called `equalize_chart()`. This method takes a **color** image, and will display a plot containing:

- the original image,
- the equalized image,
- the original image histogram, and
- the equalized image histogram.

*Tip: openCV implements histogram equalization in [cv2.equalizeHist()](https://docs.opencv.org/2.4/modules/imgproc/doc/histograms.html?highlight=equalizehist)*

In [ ]:
# ASSIGNMENT 2
# Implement a function that:
# -- converts the input image from the BGR to the YCrCb color space
# -- applies the histogram equalization
# -- displays in a 2x2 plot: the input image, the equalized one, the original histogram and the equalized one, if verbose = True
def equalize_chart(image, verbose=False):
    """ Applies histogram equalization to an image and shows the result.

        Args:
            image: Input image
            verbose: Only show images if this is True

        Returns:
            out_image: Equalized histogram image
    """

    #Transform image to YCrCb color space
    image_ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    # Create a copy of the YCrCb image to store the output, maintaining original chrominance channels.
    out_image = np.copy(image_ycrcb)

    # Apply histogram equalization to first band of the YCrCb image
    out_image[:,:,0] = cv2.equalizeHist(image_ycrcb[:,:,0])

    if verbose:

        # Plot histogram of gray image (Y channel of original)
        plt.subplot(2,2,3)
        # Use ravel() to flatten the Y channel for histogram computation and display.
        plt.hist(image_ycrcb[:,:,0].ravel(), 256, [0,256])
        plt.title('Original histogram')

        # Plot equalized histogram of the processed image (Y channel of output)
        plt.subplot(2,2,4)
        # Use ravel() to flatten the Y channel of the equalized image for histogram computation and display.
        plt.hist(out_image[:,:,0].ravel(), 256, [0,256])
        plt.title('Equalized histogram')

        # Reconvert image to RGB for displaying with matplotlib
        # Convert the original YCrCb image back to RGB for display.
        image_rgb = cv2.cvtColor(image_ycrcb, cv2.COLOR_YCrCb2RGB)
        # Convert the processed YCrCb image back to RGB for display.
        out_image_rgb = cv2.cvtColor(out_image, cv2.COLOR_YCrCb2RGB)

        # Show the initial image
        plt.subplot(2,2,1)
        plt.imshow(image_rgb)
        plt.title('Original image')

        # Show the resultant one
        plt.subplot(2,2,2)
        plt.imshow(out_image_rgb)
        plt.title('Equalized histogram image')

    return out_image

You can use the next code to **test if your results are correct**:

In [ ]:
image = np.array([[[10,60,20],[60,22,74],[72,132,2]],[[11,63,42],[36,122,27],[37,113,30]],[[1,6,2],[6,22,7],[7,13,0]]], dtype=np.uint8)
print(equalize_chart(image))

[[[128 112 110]
  [128 151 138]
  [255  68 120]]

 [[159 122 105]
  [223  87 101]
  [191  92 104]]

 [[  0 127 126]
  [ 64 122 122]
  [ 32 122 127]]]


<font color='blue'>**Expected output:**  </font>

    [[[128 112 110]
      [128 151 138]
      [255  68 120]]

     [[159 122 105]
      [223  87 101]
      [191  92 104]]

     [[  0 127 126]
      [ 64 122 122]
      [ 32 122 127]]]

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

We have developed our second image enhancement technique! Now try `equalize_chart()` with the `park.png` image in the code cell below. Then, **answer following questions**:

- What is the difference between the original histogram and the equalized one?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Is the final histogram uniform? why?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   

In [ ]:
# Read image
image = cv2.imread(images_path + 'park.png',-1)

# Equalize its histogram
interactive(equalize_chart, image=fixed(image), verbose=fixed(True))

interactive(children=(Output(),), _dom_classes=('widget-interact',))

## 2.3.3 Contrast Limited Adaptive Histogram Equalization (CLAHE) <a id=233></a>

In the previous section we explored global **histogram equalization**, which calculates a single transformation for the entire image. However, when an image contains regions with very different lighting conditions (e.g. bright backgrounds and dark foregrounds), global equalization often over-enhances bright areas and amplifies unwanted background noise.

To overcome this limitation, **Adaptive Histogram Equalization (AHE)** divides the image into small contextual regions called **tiles** (e.g. $8\times8$ pixels grids) and equalizes each tile locally. However, in relatively homogeneous areas, noise can still be over-amplified.

**CLAHE (Contrast Limited Adaptive Histogram Equalization)** solves this issue by clipping the histogram of each tile at a predefined threshold (**`clipLimit`**) before calculating the cumulative distribution function (CDF). The clipped pixels are redistributed uniformly among all histogram bins, limiting contrast enhancement and avoiding noise amplification. Finally, **bilinear interpolation** is applied along the tile borders to remove artificial boundaries.

<center><img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_processing/images/clahe_theory.png" width="650" /></center>

### **<span style="color:green"><b><i>ASSIGNMENT 3: Local enhancement with CLAHE</i></b></span>**

Similarly to the previous exercises, **you are asked to** develop a function called `clahe_chart()`. In order to compare CLAHE with global histogram equalization, this method takes a **color** image and will display a 2x3 plot containing:

- the original, globally equalized, and CLAHE-enhanced images, and
- their corresponding histograms.

*Tip: OpenCV implements CLAHE in [cv2.createCLAHE()](https://docs.opencv.org/master/d6/dc7/group__imgproc__hist.html#ga7e54071f01f3b84636b727023a485970)*

In [ ]:
    # ASSIGNMENT 3
# Implement a function that:
# -- converts the input image from the BGR to the YCrCb color space
# -- creates the CLAHE object with the given clip_limit and tile_grid_size
# -- applies CLAHE to the first band of the YCrCb image
# -- displays in a 2x3 plot: the original, globally equalized, and CLAHE-enhanced images along with their histograms, if verbose = True
def clahe_chart(image, clip_limit=2.0, tile_grid_size=(8,8), verbose=False):
    """ Applies CLAHE to an image and shows the result.

        Args:
            image: Input image
            clip_limit: Threshold for contrast limiting
            tile_grid_size: Size of grid for histogram equalization
            verbose: Only show images if this is True

        Returns:
            out_image: CLAHE enhanced image
    """

    #Transform image to YCrCb color space
    # Convert BGR to YCrCb color space.
    image_ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    # Create a copy of the YCrCb image to store the CLAHE output.
    out_image = np.copy(image_ycrcb)

    # Create CLAHE object
    # Initialize CLAHE with provided clip_limit and tile_grid_size parameters.
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    # Apply CLAHE to first band of the YCrCb image
    # Apply CLAHE only to the Y channel (luminance).
    out_image[:,:,0] = clahe.apply(image_ycrcb[:,:,0])

    if verbose:
        # Apply global histogram equalization for comparison
        equalized_image = np.copy(image_ycrcb)
        equalized_image[:,:,0] = cv2.equalizeHist(image_ycrcb[:,:,0])

        # Plot histograms
        plt.subplot(2,3,4)
        plt.hist(image_ycrcb[:,:,0].ravel(), 256, range=[0, 256])
        plt.title('Original histogram')

        plt.subplot(2,3,5)
        plt.hist(equalized_image[:,:,0].ravel(), 256, range=[0, 256])
        plt.title('Equalized histogram')

        plt.subplot(2,3,6)
        plt.hist(out_image[:,:,0].ravel(), 256, range=[0, 256])
        plt.title('CLAHE histogram')

        # Reconvert images to RGB
        image_rgb = cv2.cvtColor(image_ycrcb, cv2.COLOR_YCrCb2RGB)
        equalized_image_rgb = cv2.cvtColor(equalized_image, cv2.COLOR_YCrCb2RGB)
        out_image_rgb = cv2.cvtColor(out_image, cv2.COLOR_YCrCb2RGB)

        # Show the images
        plt.subplot(2,3,1)
        plt.imshow(image_rgb)
        plt.title('Original image')

        plt.subplot(2,3,2)
        plt.imshow(equalized_image_rgb)
        plt.title('Global equalization')

        plt.subplot(2,3,3)
        plt.imshow(out_image_rgb)
        plt.title('CLAHE')

    return out_image

You can use the next code to **test if your results are correct**:

In [ ]:
image = np.array([[[10,60,20],[60,22,74],[72,132,2]],[[11,63,42],[36,122,27],[37,113,30]],[[1,6,2],[6,22,7],[7,13,0]]], dtype=np.uint8)
print(clahe_chart(image))

[[[255 112 110]
  [255 151 138]
  [255  68 120]]

 [[255 122 105]
  [255  87 101]
  [128  92 104]]

 [[128 127 126]
  [128 122 122]
  [ 64 122 127]]]


<font color='blue'>**Expected output:**  </font>

    [[[255 112 110]
      [255 151 138]
      [255  68 120]]

     [[255 122 105]
      [255  87 101]
      [128  92 104]]

     [[128 127 126]
      [128 122 122]
      [ 64 122 127]]]

### <font color="blue"><b><i>Thinking about it (3)</i></b></font>

Now try `clahe_chart()` with the `park.png` image in the interactive code cell below. Then, **answer the following questions**:

- How does the CLAHE result compare to the global histogram equalization from the previous section? Does it look more natural?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What happens to the background noise and the local contrast as you increase the `clipLimit` parameter?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

In [ ]:
# Create widget object
clip_widget = widgets.FloatSlider(value=2.0, min=1.0, max=10.0, step=0.5, description='Clip limit:')

# Read image
image = cv2.imread(images_path + 'park.png',-1)

# Interact with your code!
interactive(clahe_chart, image=fixed(image), clip_limit=clip_widget, tile_grid_size=fixed((8,8)), verbose=fixed(True))

interactive(children=(FloatSlider(value=2.0, description='Clip limit:', max=10.0, min=1.0, step=0.5), Output()…

## <font color="orange"><b><i>AI Appendix</i></b></font>

### A.1 How I used AI
- **Model/tool:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Gemini </i></p>
- **What I asked AI to help with (bullets):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i> Explicarme los metodos que se usan </i></p>
- **Best prompt I used (paste):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>"explicame paso a paso cómo procesar solo la luminancia de una imagen python usando OpenCV usando YCrCb"</i></p>
- **What I kept vs. changed from the AI output:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Cambie códigos para que funcionasen tras un error, por errores al poner argumentos, por ejemplo.</i></p>

### A.2 AI review
Ask one generative AI to review your memo using this prompt:

```
Act as a technical reviewer of this computer vision lab notebook. Read my memo and return exactly 5 bullet points:

- two on theory (key ideas),
- two on practice (relevant programming concepts of python used, not particular functions or variables),
- one on how to improve the code (most critical aspect: limitations, parameter choices, runtime/robustness, possible bugs).

Keep each bullet ≤15 words and make them specific to my work.
```

Paste its **5** items here:

- **(Theory 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Theory 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Improvement)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

Finally, **do you agree with them?**, **would you have selected others?**
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## Conclusion

Great! We are sure that UMA users are going to appreciate your efforts. Also, next time you use an image editor tool you are going to have another point of view of how things work.

In conclusion, in this notebook you have learned:

- How to define a **gamma correction (non-linear) LUT** and to how to apply it to an image.
- How **histogram equalization** works and its applications.
- How **CLAHE** improves upon global histogram equalization by enhancing contrast locally while limiting noise amplification.

## Extra

But this doesn't have to be the end, open GIMP and look through other implemented methods.  

As you are learning about image processing, **comment how you think they are implemented from scratch.**